<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L36_Advanced_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L36: Advanced Optimization - AdamW, Schedules, Clipping

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 36 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Use AdamW with weight decay and compare to Adam
- Apply learning rate schedules (linear, cosine) with warmup
- Use gradient clipping and compare optimizer/schedule choices

---

## Concept: Advanced Optimization

**AdamW**: decoupled weight decay (better regularization). **Schedules**: warmup then decay (linear/cosine) for stable training. **Gradient clipping**: cap gradient norm to avoid explosions. Trainer exposes learning_rate, warmup_ratio, max_grad_norm, lr_scheduler_type.

---

In [ ]:
!pip install transformers torch datasets -q
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: TrainingArguments - LR, Warmup, Clipping

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(200))
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
args = TrainingArguments(
    output_dir="./out_opt_l36",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()
print("Cosine schedule + warmup + gradient clipping.")

## Part 2: Custom Scheduler (Linear with Warmup)

---

In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

model2 = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
optimizer = AdamW(model2.parameters(), lr=2e-5, weight_decay=0.01)
loader = DataLoader(train_ds, batch_size=8)
num_steps = len(loader) * 2
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*num_steps), num_training_steps=num_steps)
print(f"Warmup steps: {int(0.1*num_steps)}, total: {num_steps}")

## Exercises

1. Compare lr_scheduler_type: linear vs cosine vs constant_with_warmup; plot LR over steps.
2. Ablate max_grad_norm: 0.5, 1.0, 5.0; check stability.
3. Try different weight_decay (0, 0.01, 0.1) and report eval accuracy.

---

## Key Takeaways

1. AdamW + weight decay + warmup + decay schedule is standard for transformers.
2. max_grad_norm=1.0 prevents gradient explosion.
3. Cosine decay often gives slightly better final loss than linear.

---

## Next Lesson

**L37: Data Preparation** – Cleaning, filtering, deduplication.

---